# LLM prompting for text classification

One of the most common forms of text classification is sentiment analysis, which assigns a label like “positive”, “negative”, or “neutral” to a sequence of text. Let’s write a prompt that instructs the model to classify a given text (a movie review). We’ll start by giving the instruction, and then specifying the text to classify.

First, let's mount the Google Drive again.


In [1]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/MyDrive/2024-TU-Ilmenau

Mounted at /content/gdrive
[Errno 2] No such file or directory: '/content/gdrive/MyDrive/2024-TU-Ilmenau'
/content


In [1]:
# prompt: check if cuda is available

import torch

if torch.cuda.is_available():
  print("CUDA is available!")
else:
  print("CUDA is not available.")

CUDA is not available.


## Basic example

The Huggingface portal provides thousands of models and new ones are reselased frequentyl. In November 2024, these are some recommended smaller models to try:
* https://huggingface.co/microsoft/Phi-3.5-mini-instruct
* https://huggingface.co/Qwen/Qwen2.5-7B-Instruct
* https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.1
* [... your choice here ]

We load the model and a text generation pipeline from the HF library.

In [2]:
from transformers import pipeline, AutoTokenizer
import torch

# set seed for reproducibility
torch.manual_seed(0)

model = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model)

# tokenizer.apply_chat_template()

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
    device="cuda"
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

The LLM usually gets prompted with a system instruction and a user specific request (for chat applications the LLM is trained with the roles of a user and an assistant who engage in a dialog.). Given that prompt, the model generates a new sequence of tokens which can be configured for a maximum length. Other parameters steer the search space for optimal text generation (e.g. choosing from  top_k alternatives, generating more freely or more closely to the deterministic optmimim with lower or higher temperature)

In [7]:
# define completion function
def get_completion(input, temperature=0.7):
  messages = [
    {"role": "system", "content": "You are an expert Physicist. You are good at explaining Physics concepts in simple words. Help as much as you can."},
    {"role": "user", "content": f"{input}"}
  ]
  messages = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt=True)

  print(messages)

  response = generator(prompt,
                         max_new_tokens=100,
                         do_sample=True,
                         top_k=10,
                         num_return_sequences=1,
                         eos_token_id=tokenizer.eos_token_id,
                         temperature=temperature,
                         )
  return response

Let's play a bit.

In [8]:
# let's prompt
prompt = "Explain to me the difference between nuclear fission and fusion."
# prompt = "Why is the Sky blue?"

# try modifying the temperature. what do you observe?
response = get_completion(prompt, temperature=1.0)
print(response[0]['generated_text'])

<|system|>
You are an expert Physicist. You are good at explaining Physics concepts in simple words. Help as much as you can.<|end|>
<|user|>
Explain to me the difference between nuclear fission and fusion.<|end|>
<|assistant|>



KeyboardInterrupt: 

## Offensive language detection

We adapt this approach to perform text classification

In [ ]:
# define completion function
def detect_old(input, temperature=0.1):
  prompt = f"""
Task: Offensive Language Detection
Classify the given text into one of the following classes:
- OFFENSE: Contains any form of offensive language, including insults, defamation, slander, libel, abuse, profanity, or any degrading or disrespectful content.
- OTHER: Contains neutral, civil, or non-offensive language. This includes polite disagreement or factual statements without disrespect.

Text: "{input}"
Class: """
  response = generator(prompt,
                  max_new_tokens=10,
                  do_sample=True,
                  top_k=10,
                  num_return_sequences=1,
                  eos_token_id=tokenizer.eos_token_id,
                  temperature=temperature,
                  )
  new_tokens = response[0]['generated_text'][len(prompt):].strip()
  return new_tokens

Try some examples

In [ ]:
texts = ["You are a person to me.", "The sun is shining", "I can't stand how you talk to people like that.", "Jane hates you fuking priggs."]

What's the models answer?

In [ ]:
labels = []
for text in texts:
  response = detect_old(text)
  labels.append(response)
print(labels)

['Output: OTHER\n\nText', 'Answer: OTHER', '# Answer: OTHER', 'Answer: OFFENSE']


## Try on German tweets

Let's evaluate the GermEval 2018 test set again. At least parts of it.

In [ ]:
import pandas as pd
test_data = pd.read_csv("germeval2018.test.tsv", sep = "\t", header=None)
test_data.columns = ["text", "label", "fine"]
test_data = test_data.drop(columns=["fine"])
test_data.head()

,text,label
0,"Meine Mutter hat mir erzählt, dass mein Vater ...",OTHER
1,@Tom174_ @davidbest95 Meine Reaktion; |LBR| Ni...,OTHER
2,"#Merkel rollt dem Emir von #Katar, der islamis...",OTHER
3,„Merle ist kein junges unschuldiges Mädchen“ K...,OTHER
4,@umweltundaktiv Asylantenflut bringt eben nur ...,OFFENSE


In [ ]:
from tqdm.notebook import tqdm

tweets = test_data["text"][:200].to_list()

# This code is not optimized well. We could make use of the datasets library to improve it.
labels = []
for text in tqdm(tweets):
  response = detect_old(text)
  labels.append(response)

  0%|          | 0/200 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


The answers look a bit weird as the model generated more than than just the class name.

In [ ]:
labels

['Answer: OTHER\n\nThe',
 'Answer: OFFENSE',
 'Answer: OFFENSE',
 'Output: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Output: OTHER\n\nThe',
 'Output: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Output: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Answer: OFFENSE',
 'Answer: OFFENSE',
 'Output: OTHER\n\nThe',
 'Output: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Answer: OFFENSE',
 'Output: OFFENSE',
 'Answer: OFFENSE',
 'Answer: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Output: OTHER\n\nThe',
 'Output: OTHER\n\nThe',
 'Output: OFFENSE',
 'Output: OTHER\n\nThe',
 'Output: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Output: OTHER\n\nEx',
 'Answer: OTHER\n\nThe',
 'OFFENSE\n\nText: "The',
 'Answer: OFFENSE',
 'Answer: OTHER\n\nThe',
 '# Answer: OTHER',
 'Answer: OFFENSE',
 'Output: OFFENSE',
 'Output: OTHER\n\nEx',
 'Answer: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Output: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Answer: OTHER\n\nThe',
 'Answer: O

So, we clean the labels and remove the unwanted parts.

In [ ]:
clean_labels = ["OFFENSE" if "OFFENSE" in label else "OTHER" for label in labels]

['OTHER',
 'OFFENSE',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OFFENSE',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OFFENSE',
 'OTHER',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OFFENSE',
 'OTHER',
 'OFFENSE',
 'OFFENSE',
 'OTHE

Let's get a classification report.

In [ ]:
from sklearn.metrics import classification_report

test_labels = test_data["label"][:200].to_list()
predicted_labels = clean_labels

# Generate classification report
print(classification_report(test_labels, predicted_labels))

              precision    recall  f1-score   support

     OFFENSE       0.76      0.52      0.62        73
       OTHER       0.77      0.91      0.83       127

    accuracy                           0.77       200
   macro avg       0.76      0.71      0.72       200
weighted avg       0.76      0.77      0.75       200



# Quantization

### What is Quantization?

Quantization is a technique used to reduce the precision of the numbers (typically floating-point) used to represent a machine learning model's weights and activations. Instead of using the standard 32-bit or 16-bit floating-point values, quantization reduces them to lower-bit formats like 8-bit, 4-bit, or even 6-bit integers.

### Why is Quantization Useful for LLMs?

1. **Reduced Memory Footprint:**
   - Large language models (LLMs) like GPT-3 or Qwen-7B have billions of parameters. Quantization significantly reduces the memory required to store these models, making it feasible to run them on resource-constrained devices or hardware.

2. **Faster Inference:**
   - Lower-bit computations are computationally cheaper and faster. This allows for quicker model execution, which is critical for latency-sensitive applications.

3. **Hardware Compatibility:**
   - Quantization aligns with specialized hardware like GPUs, TPUs, and NPUs, which are optimized for lower-bit arithmetic, leading to better hardware utilization.

4. **Energy Efficiency:**
   - By reducing computational demands, quantization decreases energy consumption, making large-scale deployments more sustainable and cost-effective.

5. **Feasibility for Edge Deployment:**
   - Quantized models can run on edge devices (e.g., smartphones or IoT devices) with limited computational power, expanding the accessibility of LLMs.

### Trade-offs:
- Quantization can introduce slight accuracy degradation, especially for tasks that are highly sensitive to numerical precision. However, techniques like mixed-precision quantization and careful tuning often mitigate these effects.

By balancing efficiency and performance, quantization is an indispensable tool for scaling and deploying LLMs effectively.

In [ ]:
!pip install -U bitsandbytes

### Example of how to load a quantized model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig

# Define the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,  # Set for manual control
)

# Load the tokenizer
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model with the quantization config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto"  # Ensure automatic placement of the model on the correct device
)

# Test model inference
input_text = "Was ist der Sinn des Lebens?"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=500)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Was ist der Sinn des Lebens? Der Sinn des Lebens ist ein philosophisches und existenzielles Thema, das von verschiedenen Perspektiven betrachtet werden kann. Es gibt keine eindeutige Antwort auf diese Frage, da sie stark vom Einzelfall, den Gla


In [ ]:
# create a bit more text ...

outputs = model.generate(**inputs, max_new_tokens=500)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Was ist der Sinn des Lebens? Ein guter Weg, damit anzufangen, um das zu verstehen, ist es zu erkunden, was für uns als Menschen wichtig ist. Welche Ziele haben wir im Leben und welche Werte leiten unser Handeln? Es gibt viele verschiedene Antworten auf diese Frage, und sie können von Person zu Person stark variieren. Es gibt jedoch einige grundlegende philosophische Ansätze, die oft diskutiert werden:

1. **Selbstverwirklichung**: Viele Menschen suchen nach Bedeutung in ihrem Leben, indem sie ihre Talente und Fähigkeiten ausbauen und ihren Potenzial voll ausschöpfen. Dies kann durch kreative Tätigkeiten, berufliche Erfolge oder persönliche Herausforderungen erreicht werden.

2. **Zufriedenheit und Glücksfindung**: Andere finden den Sinn ihres Lebens darin, glücklich und zufrieden zu sein. Dies kann durch positive Beziehungen, gesundes Lebensstil und das Finden von Freude in kleinen Dingen erreicht werden.

3. **Soziale Verantwortung**: Einige Menschen finden Bedeutung, indem sie sich a

## Chat templates

When using a model like Qwen2.5 for text classification, the format of the prompt is crucial, even if you are not using it in a chat-specific use case. Many LLMs, particularly those fine-tuned on instruction-following tasks or chat-based interactions, expect inputs to follow certain conventions learned during fine-tuning. Ignoring these conventions can result in suboptimal performance. Here's what you should consider:

---

### 1. **Check the Model's Documentation**
If Qwen2.5 or its tokenizer provides a specific template (e.g., chat format with roles like `user` and `assistant`), it’s best to adhere to it even for text classification. These templates often include separators, prefixes, or structures that the model relies on for contextual understanding.

#### Example (Hypothetical Chat Template):
```plaintext
User: Classify the following text:
"I can't believe this happened."

Assistant: The text is classified as:
```

While this might seem redundant for a simple text classification task, it ensures the model operates in the expected input-output structure.

---

### 2. **Adapt the Prompt to a Non-Chat Use Case**
If the chat format doesn't suit your task, you can simplify the prompt but still follow the instruction-tuned structure. The key is to maintain clarity in task description and output expectations.

#### Example (Simplified Instruction Prompt):
```plaintext
Task: Text Classification
Classify the text into one of the following categories:
- OFFENSE
- OTHER

Text: "I can't believe this happened."
Class:
```

---

### 3. **Experiment with Prompt Variations**
LLMs like Qwen2.5 are versatile and often robust to prompt variations. You can experiment with:
- Adding explicit task instructions.
- Including examples (few-shot learning).
- Testing shorter or more direct prompts.

#### Few-Shot Example:
```plaintext
Task: Text Classification
Classify the text into one of the following categories:
- OFFENSE
- OTHER

Examples:
1. "You are so annoying." -> OFFENSE
2. "This is a fantastic discovery." -> OTHER

Text: "I can't believe this happened."
Class:
```

---

### 4. **Why Stick to Chat Templates?**
Even if your task isn't chat-based:
- **Tokenization Dependencies:** Some models tokenize differently for chat inputs versus plain text, affecting performance.
- **Fine-Tuning Behavior:** Models fine-tuned with chat structures may have learned to interpret queries better within those contexts.

If ignoring the chat format, ensure your prompt includes explicit task definitions and expected outputs to compensate.

---

### 5. **Implementation and Testing**
- **Start Simple:** Use direct prompts and observe the model’s responses.
- **Refine Based on Performance:** If the outputs are inconsistent or biased, revert to the chat template or add clarifications in the prompt.
- **Test Across Variations:** Experiment with multiple formats to find the most reliable for your dataset and task.

---

### Conclusion
While you don't necessarily have to use the exact chat templates, you should aim to align your prompts with the model's training paradigm. Clear, instruction-following prompts generally work well, but adhering to any provided templates can improve reliability, especially for models fine-tuned on chat data.

# Optional exercise

Evaluate the Qwen2.5 model with the GermEval 2018 dataset.